# Library

In [2]:
import plotly.graph_objs as pgo
from plotly.subplots import make_subplots

In [3]:
import ipywidgets as widgets
from IPython.display import display

In [6]:
from collections import deque

# Real-time Plotter MultiChannel

In [5]:
PLOTTER_WINDOW_WIDTH_DP_DEFAULT = 600   
PLOTTER_WINDOW_HEIGH_PT_DEFAULT = 120 

class PlotterMultiChannel:
    def __init__(self, aChannelList, 
                 aWinWidth=PLOTTER_WINDOW_WIDTH_DP_DEFAULT):
        self._win_width_dp = aWinWidth
        self._traces = {'tick': (0, deque(maxlen=self._win_width_dp))}
        
        self._init_layout(aChannelList)
        display(self._fig)


    def _init_fig(self, aChannelList):
        num_channels = len(aChannelList)        
        self._fig = pgo.FigureWidget(make_subplots(rows=num_channels, cols=1, 
                                                   shared_xaxes=True, 
                                                   vertical_spacing=0.02))
        self._fig.update_layout(
            title=f"Multi-channel Plotter (WinWidth={self._win_width_dp} dp, Freq<12 fps)",
            title_x=0.5,
            showlegend=False,
            height=PLOTTER_WINDOW_HEIGH_PT_DEFAULT*(num_channels+1)  
        )
        
        for idx, name in enumerate(aChannelList, 1):
            self._fig.add_scatter(x=[0], y=[0], 
                                  mode='lines', name=name, 
                                  row=idx, col=1)
            self._fig.update_yaxes(title_text=name, row=idx, col=1)
            self._traces[name] = (idx-1, deque(maxlen=self._win_width_dp))
            self._traces_key_cache = set(self._traces.keys())

    def _init_layout(self, aChannelList):
        self._init_fig(aChannelList)


    def ingest(self, aDataDict):  # {'tick': int, 'metrics': {'metric_1': float, 'metric_2': float, ...}}
        new_tick = aDataDict['tick']
        if len(self._traces['tick'][1])>0:
            if not (new_tick > self._traces['tick'][1][-1]): 
                self._on_btn_click_reset()
                
        self._traces['tick'][1].append(new_tick)
        for curr_name, curr_value in aDataDict['metrics'].items():
            if curr_name in self._traces_key_cache:
                self._traces[curr_name][1].append(curr_value)

        with self._fig.batch_update():
            for curr_name in aDataDict['metrics'].keys():
                if curr_name in self._traces_key_cache:
                    trace = self._fig.data[self._traces[curr_name][0]]
                    trace.x = list(self._traces['tick'][1])
                    trace.y = list(self._traces[curr_name][1])            
            
            if not (len(self._traces['tick'][1]) < self._win_width_dp):
                self._fig.update_xaxes(range=[self._traces['tick'][1][0]-1, 
                                              self._traces['tick'][1][-1]+1], 
                                       row=1, col=1) #given shared x-axes, only need to update one


    def ingest_str(self, aDictStr):
        self.ingest(eval(aDictStr))